# 01 — Data Exploration & Subsample Index Generation

This notebook performs:
1. Dataset loading and EDA across 4 languages (English, Hindi, Arabic, Spanish)
2. Class distribution analysis
3. Text length and token length statistics
4. **Stratified subsample index generation** (`subsample_indices.json`)

⚠️ **The subsample indices file is generated ONCE here and committed to git. All evaluation runs load from this file.**

In [ ]:
# Cell 1: Setup
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from collections import Counter

from src.dataset import (
    load_all_splits, LANGUAGES, LABEL_MAP,
    get_stratified_subsample, save_subsample_indices, load_subsample_indices
)

sns.set_theme(style='whitegrid', palette='muted')
print('Setup complete.')

In [ ]:
# Cell 2: Load all datasets
data = load_all_splits(LANGUAGES)

for lang in LANGUAGES:
    splits = data[lang]
    print(f"\n{lang.upper()}:")
    for split_name, ds in splits.items():
        print(f"  {split_name}: {len(ds)} samples")

In [ ]:
# Cell 3: Class distribution per language and split
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, lang in enumerate(LANGUAGES):
    ax = axes[idx // 2][idx % 2]
    
    distributions = {}
    for split_name in ['train', 'validation', 'test']:
        labels = data[lang][split_name]['label']
        counts = Counter(labels)
        distributions[split_name] = {
            LABEL_MAP[k]: v for k, v in sorted(counts.items())
        }
    
    df = pd.DataFrame(distributions)
    df.plot(kind='bar', ax=ax)
    ax.set_title(f'{lang.capitalize()} — Label Distribution', fontweight='bold')
    ax.set_xlabel('Sentiment')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Text length statistics (characters and words)
stats_rows = []

for lang in LANGUAGES:
    for split_name in ['train', 'test']:
        texts = data[lang][split_name]['text']
        char_lengths = [len(t) for t in texts]
        word_lengths = [len(t.split()) for t in texts]
        
        stats_rows.append({
            'Language': lang.capitalize(),
            'Split': split_name,
            'Samples': len(texts),
            'Avg Chars': f"{np.mean(char_lengths):.1f}",
            'Max Chars': max(char_lengths),
            'Avg Words': f"{np.mean(word_lengths):.1f}",
            'Max Words': max(word_lengths),
        })

stats_df = pd.DataFrame(stats_rows)
print(stats_df.to_string(index=False))

In [ ]:
# Cell 5: Token length analysis with Qwen2.5 tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-3B-Instruct')

token_stats = []
for lang in LANGUAGES:
    texts = data[lang]['test']['text']
    token_lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in texts]
    word_lengths = [len(t.split()) for t in texts]
    fertility = sum(token_lengths) / max(sum(word_lengths), 1)
    
    token_stats.append({
        'Language': lang.capitalize(),
        'Avg Tokens': f"{np.mean(token_lengths):.1f}",
        'Max Tokens': max(token_lengths),
        'Fertility': f"{fertility:.3f}",
    })
    print(f"{lang}: avg={np.mean(token_lengths):.1f}, max={max(token_lengths)}, fertility={fertility:.3f}")

pd.DataFrame(token_stats)

In [ ]:
# Cell 6: Token length distribution plot
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for idx, lang in enumerate(LANGUAGES):
    texts = data[lang]['test']['text']
    token_lengths = [len(tokenizer.encode(t, add_special_tokens=False)) for t in texts]
    
    axes[idx].hist(token_lengths, bins=30, edgecolor='white', alpha=0.8, color=f'C{idx}')
    axes[idx].set_title(f'{lang.capitalize()}', fontweight='bold')
    axes[idx].set_xlabel('Token Count')
    axes[idx].axvline(x=512, color='red', linestyle='--', alpha=0.7, label='max_seq_len=512')

axes[0].set_ylabel('Frequency')
axes[-1].legend()
plt.suptitle('Token Length Distribution per Language (Qwen2.5 tokenizer)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Sample texts from each language
for lang in LANGUAGES:
    print(f"\n{'='*60}")
    print(f"{lang.upper()} - Sample test texts")
    print('='*60)
    for i in range(3):
        text = data[lang]['test'][i]['text']
        label = LABEL_MAP[data[lang]['test'][i]['label']]
        print(f"  [{label:>8}] {text[:100]}{'...' if len(text) > 100 else ''}")

# Confirm ChatML formatting before training
from src.dataset import format_for_sft, get_label_map

label_map, _ = get_label_map()
sample = data["english"]["train"][0]
formatted = format_for_sft(sample, tokenizer)
chatml_text = formatted["formatted_text"]

print("\n" + "=" * 72)
print("CHATML FORMATTING CHECK")
print("=" * 72)
print(f"Label in sample: {label_map[sample['label']]}")
print("\nRaw ChatML text:")
print("-" * 72)
print(chatml_text)

# Pretty preview: each ChatML block starts on a new line for readability.
pretty_chatml = (
    chatml_text
    .replace("<|im_start|>", "\n<|im_start|>")
    .replace("<|im_end|>", "<|im_end|>\n")
)

print("\nPretty ChatML preview:")
print("-" * 72)
print(pretty_chatml.strip())
print("=" * 72)

In [ ]:
# Cell 8: Hindi script composition (Devanagari vs Romanized)
import json
import unicodedata
from pathlib import Path


def has_devanagari(text: str) -> bool:
    return any(
        "DEVANAGARI" in unicodedata.name(c, "")
        for c in text
    )


def script_composition(dataset, language_name: str) -> dict:
    total = len(dataset)
    devanagari_count = sum(
        1 for ex in dataset if has_devanagari(ex["text"])
    )
    roman_count = total - devanagari_count
    print(f"\n{language_name} script composition ({total} samples):")
    print(f"  Devanagari script: {devanagari_count:4d} "
          f"({devanagari_count/total*100:.1f}%)")
    print(f"  Romanized/Latin:   {roman_count:4d} "
          f"({roman_count/total*100:.1f}%)")
    return {
        "devanagari": devanagari_count,
        "roman": roman_count,
        "total": total,
        "devanagari_pct": round(devanagari_count/total*100, 1)
    }


# Run on Hindi train and test splits
hindi_test = data["hindi"]["test"]
hindi_train = data["hindi"]["train"]

test_composition = script_composition(hindi_test, "Hindi Test")
train_composition = script_composition(hindi_train, "Hindi Train")

# Interpretation
if test_composition["devanagari_pct"] < 30:
    print("\n⚠️  PAPER NOTE: Hindi dataset is predominantly Romanized.")
    print("   Token fertility (1.703) reflects Latin-script Hindi,")
    print("   not Devanagari. This should be noted in paper Discussion.")
    print("   Frame as: 'Hindi tweets are code-mixed/Romanized,")
    print("   explaining fertility closer to English than expected.'")
elif test_composition["devanagari_pct"] > 70:
    print("\n✅ Hindi dataset is predominantly Devanagari script.")
    print("   Token fertility argument holds as stated.")
else:
    print("\n⚠️  PAPER NOTE: Hindi dataset is mixed script.")
    print("   Report exact percentages in paper footnote.")

composition_data = {
    "test": test_composition,
    "train": train_composition,
    "finding": (
        "predominantly_romanized"
        if test_composition["devanagari_pct"] < 30
        else "predominantly_devanagari"
        if test_composition["devanagari_pct"] > 70
        else "mixed_script"
    ),
    "paper_implication": (
        "Token fertility of 1.703 reflects Latin-script tokenization. "
        "Discuss in paper as code-mixing characteristic of Hindi Twitter data."
    )
}

out_path = Path("../results/eval_scores/hindi_script_composition.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(composition_data, f, ensure_ascii=False, indent=2)

print(f"\nSaved to {out_path}")

In [ ]:
# Cell 9: Generate stratified subsample indices
# ⚠️ WARNING: This cell should only be run ONCE.
# If subsample_indices.json already exists, it will raise FileExistsError.
# This is intentional — to regenerate, delete the file manually first.

test_datasets = {lang: data[lang]['test'] for lang in LANGUAGES}

try:
    indices = save_subsample_indices(
        test_datasets,
        n_samples=200,
        seed=42,
        save_path='../results/eval_scores/subsample_indices.json'
    )
    print(f"\n✅ Subsample indices saved!")
    print(f"   Total samples: {indices['total_samples']}")
    print(f"   Seed: {indices['seed']}")
    for lang in LANGUAGES:
        print(f"   {lang}: {len(indices['indices'][lang])} indices")
        print(f"     Distribution: {indices['class_distribution'][lang]}")
except FileExistsError as e:
    print(f"ℹ️ {e}")
    indices = load_subsample_indices('../results/eval_scores/subsample_indices.json')
    print(f"   Loaded existing indices: {indices['total_samples']} total")

In [ ]:
# Cell 10: Verify subsample preservation of class distribution
indices = load_subsample_indices('../results/eval_scores/subsample_indices.json')

print("Class distribution comparison (full test vs subsample):")
print(f"{'Language':<12} {'Full Test':>10} {'Subsample':>10} {'Match?':>8}")
print('-' * 42)

for lang in LANGUAGES:
    full_dist = Counter(data[lang]['test']['label'])
    sub_dist = indices['class_distribution'][lang]
    
    full_pcts = {k: round(v / sum(full_dist.values()) * 100, 1) for k, v in sorted(full_dist.items())}
    sub_total = sum(sub_dist.values())
    sub_pcts = {k: round(v / sub_total * 100, 1) for k, v in sub_dist.items()}
    
    print(f"{lang:<12} {str(dict(sorted(full_dist.items()))):>40}")
    print(f"{'':12} {str(sub_dist):>40}  (subsample)")
    
print(f"\n✅ Subsample indices verified. Commit this file to git.")